# Fruit Deep Learning Classifier — Train & Evaluate

End-to-end pipeline: pull the [Fruit and Vegetable Disease (Healthy vs Rotten)](https://www.kaggle.com/datasets/muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten) dataset from Kaggle via `kagglehub`, then train and evaluate a transfer-learning classifier on the 28 healthy/rotten classes.

Backbone is selectable — set `BACKBONE` in the config cell to `'mobilenet_v2'` or `'efficientnet_v2_s'`.

**Kaggle auth:** `kagglehub` reads credentials from `~/.kaggle/kaggle.json` (create one via *Kaggle → Account → Create New API Token*). On first run in a Colab/Kaggle notebook it will prompt you interactively — locally, drop the file at `~/.kaggle/kaggle.json` (`%USERPROFILE%\.kaggle\kaggle.json` on Windows).

## 1. Install dependencies

In [ ]:
%pip install --quiet torch torchvision kagglehub pillow matplotlib

## 2. Config

In [ ]:
BACKBONE     = 'mobilenet_v2'   # or 'efficientnet_v2_s'
BATCH_SIZE   = 32
EPOCHS       = 3
LEARNING_RATE = 1e-3
VAL_SPLIT    = 0.2
SEED         = 42
MODEL_PATH   = 'best_model.pth'

import torch, random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 3. Download dataset from Kaggle

`kagglehub` caches the dataset under `~/.cache/kagglehub/`, so subsequent runs are instant.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download('muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten')
print('Downloaded to:', dataset_path)

import os
# The dataset may nest a single top-level folder — descend if so.
def find_image_root(root):
    entries = os.listdir(root)
    subdirs = [e for e in entries if os.path.isdir(os.path.join(root, e))]
    # If every entry is a class folder (contains images), we're done.
    if subdirs and any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in os.listdir(os.path.join(root, subdirs[0]))):
        return root
    if len(subdirs) == 1:
        return find_image_root(os.path.join(root, subdirs[0]))
    return root

image_root = find_image_root(dataset_path)
print('Image root:', image_root)
print('Classes:', sorted(os.listdir(image_root))[:6], '...')

## 4. DataLoaders (train / val split)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

train_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_ds = datasets.ImageFolder(image_root, transform=train_tfm)
class_names = full_ds.classes
print(f'Found {len(class_names)} classes, {len(full_ds)} images')

val_size = int(len(full_ds) * VAL_SPLIT)
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
# Swap in the eval transform for the validation subset without duplicating the dataset.
val_ds.dataset = datasets.ImageFolder(image_root, transform=eval_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}')

## 5. Build the model (transfer learning)

Load a pretrained backbone, freeze its features, replace the classifier head with a fresh `Linear(→ num_classes)`.

In [ ]:
import torch.nn as nn
from torchvision.models import (
    mobilenet_v2, MobileNet_V2_Weights,
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
)

def build_model(backbone: str, num_classes: int) -> nn.Module:
    if backbone == 'mobilenet_v2':
        model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    elif backbone == 'efficientnet_v2_s':
        model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
    else:
        raise ValueError(f'Unknown backbone: {backbone}')
    for p in model.features.parameters():
        p.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_model(BACKBONE, num_classes=len(class_names)).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'{BACKBONE}: {trainable:,} trainable / {total:,} total params')

## 6. Training loop

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    loss_sum = correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss_sum += criterion(out, yb).item() * xb.size(0)
        correct  += (out.argmax(1) == yb).sum().item()
        total    += yb.size(0)
    return loss_sum / total, correct / total

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = correct = total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        correct  += (out.argmax(1) == yb).sum().item()
        total    += yb.size(0)
    train_loss, train_acc = loss_sum / total, correct / total
    val_loss,   val_acc   = evaluate(val_loader)

    history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss);     history['val_acc'].append(val_acc)
    print(f'Epoch {epoch}/{EPOCHS} | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  -> saved best model (val_acc={val_acc:.4f}) to {MODEL_PATH}')

## 7. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(history['train_loss'], label='train'); ax1.plot(history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.set_xlabel('epoch'); ax1.legend()
ax2.plot(history['train_acc'],  label='train'); ax2.plot(history['val_acc'],  label='val')
ax2.set_title('Accuracy'); ax2.set_xlabel('epoch'); ax2.legend()
plt.tight_layout(); plt.show()

## 8. Inference on a single image

In [ ]:
from PIL import Image

# Load best checkpoint into a fresh model.
infer_model = build_model(BACKBONE, num_classes=len(class_names))
infer_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
infer_model.to(device).eval()

def predict(image_path: str, topk: int = 3):
    img = Image.open(image_path).convert('RGB')
    x = eval_tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(infer_model(x), dim=1)[0]
    top = torch.topk(probs, k=topk)
    return [(class_names[i.item()], v.item()) for v, i in zip(top.values, top.indices)]

# Grab a sample image from the dataset and predict it.
import glob
sample = glob.glob(os.path.join(image_root, class_names[0], '*.jp*g'))[0]
print('Sample image:', sample)
for cls, p in predict(sample):
    print(f'  {cls}: {p:.3f}')